# Assignment 3

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [4]:
# Load the libraries as required.
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import shap
import joblib

In [5]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


# Get X and Y

Create the features data frame and target data.

In [6]:
#Create the features data frame (X)
x = fires_dt.drop(columns='area')

In [7]:
#Create the target data (Y)
y = fires_dt['area']

In [8]:
print("Shape of X:", x.shape)
print("Shape of y:", y.shape)


Shape of X: (517, 12)
Shape of y: (517,)


In [45]:
X = fires_dt.drop('area', axis=1)
y = fires_dt['area']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [9]:
# Define numerical and categorical columns
numerical_cols = ['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain']
categorical_cols = ['month', 'day']

In [10]:
# Define Preprocessor 1
# Numeric: scaling only
# Categorical: one-hot encoding

preproc1 = ColumnTransformer([
    ('num', StandardScaler(), numerical_cols),          # Apply Standard Scaler to numerical columns
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)  # Apply One-Hot Encoding to categorical columns
])


### Preproc 2

Create preproc2 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [11]:
# We'll apply a log transformation to 'dmc', 'dc', and 'isi' as these often have skewed distributions
def selective_log_transform(X):
    X_copy = X.copy()
    log_columns = ['dmc', 'dc', 'isi']
    for col in log_columns:
        if col in X_copy.columns:
            X_copy[col] = np.log1p(X_copy[col])
    return X_copy

In [12]:
preproc2 = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('log_transform', ColumnTransformer(selective_log_transform)),
        ('scaler', StandardScaler())
    ]), numerical_cols),
     ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.metrics import make_scorer, mean_squared_error, r2_score, mean_absolute_error, explained_variance_score
import numpy as np


In [14]:
# Pipeline A = preproc1 + baseline
pipeline_A = Pipeline([
    ('preprocessing', preproc1),
    ('regressor', Ridge())
])

In [15]:
# Pipeline B = preproc2 + baseline
pipeline_B = Pipeline([
    ('preprocessing', preproc2),
    ('regressor', Ridge())
])

In [54]:
# Pipeline C = preproc1 + advanced model
pipeline_C = Pipeline([
    ('preprocessing', preproc1),
    ('regressor', RandomForestRegressor(random_state=42))
])

In [16]:
# Pipeline D = preproc2 + advanced model
pipeline_D = Pipeline([
    ('preprocessing', preproc2),
    ('regressor', RandomForestRegressor(random_state=42))
])
    

# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [17]:
# Ridge Regression hyperparameters
param_grid_ridge = {'regressor__alpha': [0.1, 1, 10, 100]}

# Random Forest hyperparameters
param_grid_rf = {
    'regressor__n_estimators': [50, 100, 200, 300], 
    'regressor__max_depth': [10, 20, None], 
    'regressor__min_samples_split': [2, 5, 10]
}

# Define a function to perform grid search and report results
def perform_grid_search(pipeline, param_grid, X, y, pipeline_name):
    # Create a scorer - we'll use negative mean squared error
    scorer = make_scorer(mean_squared_error, greater_is_better=False)

In [18]:
 # Perform GridSearchCV
grid_search = GridSearchCV(pipeline_A, param_grid_ridge, scoring=make_scorer, cv=5, n_jobs=-1, verbose=1)
grid_search.fit(x, y)

Fitting 5 folds for each of 4 candidates, totalling 20 fits


c:\Users\user\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\model_selection\_search.py:1102: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan]
  warnings.warn(


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(transformers=[('num',
                                                                         StandardScaler(),
                                                                         ['coord_x',
                                                                          'coord_y',
                                                                          'ffmc',
                                                                          'dmc',
                                                                          'dc',
                                                                          'isi',
                                                                          'temp',
                                                                          'rh',
                                                                          'wind',
                                                                          'rain']),
                                                                        ('cat',
                                                                         OneHotEncoder(handle_unknown='ignore'),
                                                                         ['month',
                                                                          'day'])])),
                                       ('regressor', Ridge())]),
             n_jobs=-1, param_grid={'regressor__alpha': [0.1, 1, 10, 100]},
             scoring=<function make_scorer at 0x00000212F0B78CA0>, verbose=1)

In [19]:
# GridSearch for Pipeline A (preproc1 + Ridge)
grid_search_a = GridSearchCV(pipeline_A, param_grid_ridge, cv=5, scoring='neg_mean_squared_error')
grid_search_a.fit(x, y)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(transformers=[('num',
                                                                         StandardScaler(),
                                                                         ['coord_x',
                                                                          'coord_y',
                                                                          'ffmc',
                                                                          'dmc',
                                                                          'dc',
                                                                          'isi',
                                                                          'temp',
                                                                          'rh',
                                                                          'wind',
                                                                          'rain']),
                                                                        ('cat',
                                                                         OneHotEncoder(handle_unknown='ignore'),
                                                                         ['month',
                                                                          'day'])])),
                                       ('regressor', Ridge())]),
             param_grid={'regressor__alpha': [0.1, 1, 10, 100]},
             scoring='neg_mean_squared_error')

In [20]:

# Define a function to perform grid search and report results
def perform_grid_search(pipeline, param_grid, X, y, pipeline_name):
    # Create a scorer - we'll use negative mean squared error
    scorer = make_scorer(mean_squared_error, greater_is_better=False)
    
    # Perform GridSearchCV
    grid_search = GridSearchCV(pipeline, param_grid, scoring=scorer, cv=5, n_jobs=-1, verbose=1)
    grid_search.fit(X, y)

In [21]:
# Print results
print(f"\nResults for {pipeline_A}:")
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best RMSE: {np.sqrt(-grid_search.best_score_):.4f}")
    


Results for Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi', 'temp',
                                                   'rh', 'wind', 'rain']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['month', 'day'])])),
                ('regressor', Ridge())]):
Best parameters: {'regressor__alpha': 0.1}
Best RMSE: nan


In [22]:
# Print all results
means = grid_search.cv_results_['mean_test_score']
stds = grid_search.cv_results_['std_test_score']
for mean, std, params in zip(means, stds, grid_search.cv_results_['params']):
    print(f"RMSE: {np.sqrt(-mean):.4f} (+/-{std * 2:.4f}) for {params}")
grid_search.best_estimator_


RMSE: nan (+/-nan) for {'regressor__alpha': 0.1}
RMSE: nan (+/-nan) for {'regressor__alpha': 1}
RMSE: nan (+/-nan) for {'regressor__alpha': 10}
RMSE: nan (+/-nan) for {'regressor__alpha': 100}


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['coord_x', 'coord_y', 'ffmc',
                                                   'dmc', 'dc', 'isi', 'temp',
                                                   'rh', 'wind', 'rain']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['month', 'day'])])),
                ('regressor', Ridge(alpha=0.1))])

In [23]:
# Define parameter grids for each pipeline
param_grid_A = {
    'preprocessing__num__scaler': ['standard', 'robust', 'minmax'],
    'regressor__alpha': [0.1, 1.0, 10.0, 100.0]
}


In [24]:
param_grid_B = {
    'preprocessing__num__scaler': ['standard', 'robust', 'minmax'],
    'preprocessing__num__log_transform__func': [None, np.log1p],
    'regressor__alpha': [0.1, 1.0, 10.0, 100.0]
}

In [25]:
param_grid_C = {
    'preprocessing__num__scaler': ['standard', 'robust', 'minmax'],
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [10, 20, None],
    'regressor__min_samples_split': [2, 5]
}

In [26]:
param_grid_D = {
    'preprocessing__num__scaler': ['standard', 'robust', 'minmax'],
    'preprocessing__num__log_transform__func': [None, np.log1p],
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [10, 20, None],
    'regressor__min_samples_split': [2, 5]
}

# Evaluate

+ Which model has the best performance?

In [ ]:
# Ridge Regression (Pipelines A and B)
print("Pipeline A Best Params:", grid_search_a.best_params_)
print("Pipeline A Best Score (MSE):", -grid_search_a.best_score_)

print("Pipeline B Best Params:", grid_search_b.best_params_)
print("Pipeline B Best Score (MSE):", -grid_search_b.best_score_)

# Random Forest (Pipelines C and D)
print("Pipeline C Best Params:", grid_search_c.best_params_)
print("Pipeline C Best Score (MSE):", -grid_search_c.best_score_)

print("Pipeline D Best Params:", grid_search_d.best_params_)
print("Pipeline D Best Score (MSE):", -grid_search_d.best_score_)


Pipeline A Best Params: {'regressor__alpha': 100}
Pipeline A Best Score (MSE): 4178.413267161371


In [28]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Select the best model from the GridSearch
best_pipeline = grid_search.best_estimator_

# Make predictions on the test set
Y_test_pred = best_pipeline.predict(x)

# Calculate metrics
test_rmse = np.sqrt(mean_squared_error(y, Y_test_pred))
test_mae = mean_absolute_error(y, Y_test_pred)
test_r2 = r2_score(y, Y_test_pred)

# Display results
print("Test Set Results (Pipeline D):")
print(f"RMSE: {test_rmse:.4f}")
print(f"MAE: {test_mae:.4f}")
print(f"R²: {test_r2:.4f}")


Test Set Results (Pipeline D):
RMSE: 62.1223
MAE: 20.2541
R²: 0.0458


In [62]:
# Assign best estimator
best_model = grid_search.best_estimator_

# Export

+ Save the best performing model to a pickle file.

In [29]:
import pickle
import numpy as np
from sklearn.metrics import mean_squared_error
import joblib
import shap
import pandas as pd
import matplotlib.pyplot as plt


In [57]:
# Function to evaluate model performance
def evaluate_model(model, X, y):
    y_pred = model.predict(X)
    mse = mean_squared_error(y, y_pred)
    rmse = np.sqrt(mse)
    return rmse

In [104]:
# File path for saving the model
model_filepath = './best_model_pipeline.pkl'


In [37]:
# Save the best-performing model
joblib.dump(best_pipeline, model_filepath)

print(f"Best model saved to: {model_filepath}")


Best model saved to: ./best_model_pipeline.pkl


In [38]:
# Load the saved model
loaded_model = joblib.load(model_filepath)



# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

*(Answer here.)*

In [ ]:
%pip install shap
import shap


In [86]:
# Initialize SHAP explainer for the best-performing model
explainer = shap.Explainer(best_pipeline['regressor'], best_pipeline['preprocessing'].transform(X_test))



In [47]:
# Select an observation from the test set
observation_idx = 10  # Change index as needed
observation = X_test.iloc[observation_idx:observation_idx+1]  # Single observation

In [66]:
print(observation.shape)


(1, 12)


In [94]:
import shap

# Transform training data for SHAP initialization
background_data = best_model.named_steps['preprocessing'].transform(X_train)

# Initialize the SHAP explainer with the transformed training data
explainer = shap.Explainer(best_pipeline['regressor'], best_pipeline['preprocessing'].transform(X_test))


# Transform the test data
transformed_X_test = best_model.named_steps['preprocessing'].transform(X_test)

# Get SHAP values for the transformed test data
shap_values = explainer.shap_values(transformed_X_test)


In [63]:

# Get feature names after preprocessing
feature_names = best_model.named_steps['preprocessing'].get_feature_names_out()


In [64]:
# 1. Explain a specific observation
def explain_observation(observation_index):
    print(f"Explaining observation {observation_index}")
    

In [74]:
 
# Get the specific observation
observation = X_test.iloc[observation_idx]

In [75]:
# Calculate SHAP values for this observation
observation_shap = explainer.shap_values(best_model.named_steps['preprocessing'].transform(observation.to_frame().T))[0]

In [76]:
# Create a DataFrame with feature names and SHAP values
shap_df = pd.DataFrame({
        'feature': feature_names,
        'shap_value': observation_shap
    }).sort_values('shap_value', key=abs, ascending=False)
    
print("Top 5 most important features for this observation:")
print(shap_df.head())

Top 5 most important features for this observation:
           feature  shap_value
4          num__dc  -26.082100
21  cat__month_sep  -13.866217
3         num__dmc   12.588422
11  cat__month_aug   11.838721
6        num__temp    9.147428


In [78]:
# Example: Explain the first observation in the test set
explain_observation(0)

Explaining observation 0


In [81]:
# 2. Global feature importance
def global_feature_importance():
    # Calculate mean absolute SHAP values for each feature
    mean_shap = np.abs(shap_values).mean(axis=0)

In [95]:
# 3. Feature removal analysis
def feature_removal_analysis():
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': np.abs(shap_values).mean(axis=0)
    }).sort_values('importance', ascending=True)
    
    print("Features to consider removing (least important first):")
    print(importance_df.head())
    
    print("\nTo test if these features enhance model performance:")
    print("1. Remove these features one by one, starting with the least important.")
    print("2. Retrain the model without each feature.")
    print("3. Evaluate the model's performance (e.g., RMSE) on a validation set.")
    print("4. Compare the performance to the original model with all features.")
    print("5. If performance doesn't degrade significantly, consider permanently removing the feature.")

feature_removal_analysis()

Features to consider removing (least important first):
           feature  importance
19  cat__month_nov    0.000000
27    cat__day_tue    0.033554
1     num__coord_y    0.101515
9        num__rain    0.121545
14  cat__month_jan    0.161548

To test if these features enhance model performance:
1. Remove these features one by one, starting with the least important.
2. Retrain the model without each feature.
3. Evaluate the model's performance (e.g., RMSE) on a validation set.
4. Compare the performance to the original model with all features.
5. If performance doesn't degrade significantly, consider permanently removing the feature.


## Criteria

The [rubric](./assignment_3_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-3`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_3.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.